**PubMedBERT Baseline- PubMedQA Fine-tuning and Evaluation**

In [ ]:
# IMPORTS
import json
import torch
import numpy as np
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [ ]:
# CONFIGURATION
# Model, dataset paths, training hyperparameters
PQAL_PATH  = "/content/ori_pqal (1).json"
GT_PATH    = "/content/test_ground_truth (1).json"
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
ID2LABEL   = {0: "yes", 1: "no", 2: "maybe"}
LABEL2ID   = {"yes": 0, "no": 1, "maybe": 2}
MAX_LENGTH = 512
BATCH_SIZE = 8
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cuda


In [ ]:
# LOAD DATA
# Load PubMedQA labeled dataset and ground truth test labels
with open(PQAL_PATH) as f:
    pqal = json.load(f)
with open(GT_PATH) as f:
    ground_truth = json.load(f)
# Build test samples from ground truth PMIDs

test_samples = []
for pmid, label in ground_truth.items():
    if pmid not in pqal:
        continue
    entry = pqal[pmid]
    context = " ".join(entry.get("CONTEXTS", []))
    test_samples.append({
        "pmid":     pmid,
        "question": entry["QUESTION"],
        "context":  context,
        "label":    label
    })

print(f"Test samples loaded : {len(test_samples)}")
print(f"Label distribution  : {dict(Counter(s['label'] for s in test_samples))}")


Test samples loaded : 500
Label distribution  : {'yes': 276, 'no': 169, 'maybe': 55}


In [ ]:
# BUILD TRAINING POOL
# Use all labeled PQAL instances NOT in the test set for fine-tuning
# Standard split: 450 train / 50 dev / 500 test
test_pmids = set(ground_truth.keys())
train_samples = []
for pmid, entry in pqal.items():
    if pmid in test_pmids:
        continue
    label = entry.get("final_decision", "")
    if label not in ("yes", "no", "maybe"):
        continue
    context = " ".join(entry.get("CONTEXTS", []))
    train_samples.append({
        "question": entry["QUESTION"],
        "context":  context,
        "label":    label
    })

print(f"Train pool size     : {len(train_samples)}")

Train pool size     : 500


In [ ]:
# LOAD PUBMEDBERT MODEL
# Load pretrained PubMedBERT with a randomly initialized
# 3-class classification head (yes / no / maybe)
print(f"\nLoading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)
model.to(DEVICE)
print("Model loaded.")


Loading microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- 

Model loaded.


In [ ]:
# DATASET CLASS
# PyTorch Dataset for tokenizing question + context pairs
# Input format: [CLS] question [SEP] context [SEP]
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from transformers import get_linear_schedule_with_warmup

class PubMedQADataset(Dataset):
    def __init__(self, samples, tokenizer, max_length):
        self.samples    = samples
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        enc = self.tokenizer(
            s["question"], s["context"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(LABEL2ID[s["label"]], dtype=torch.long)
        }
#CLASS WEIGHTS
#Compute inverse frequency weights to handle label imbalance
EPOCHS = 3
LR = 1e-3
label_counts = Counter(s["label"] for s in train_samples)
total = sum(label_counts.values())
print(f"Train label distribution: {dict(label_counts)}")

weights = torch.tensor([
    total / (3 * label_counts["yes"]),
    total / (3 * label_counts["no"]),
    total / (3 * label_counts["maybe"])
], dtype=torch.float).to(DEVICE)
print(f"Class weights — yes: {weights[0]:.3f}, no: {weights[1]:.3f}, maybe: {weights[2]:.3f}")

loss_fn = CrossEntropyLoss()

train_dataset = PubMedQADataset(train_samples, tokenizer, MAX_LENGTH)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)
# DataLoader, AdamW optimizer, linear warmup scheduler
#Train PubMedBERT classification head on PubMedQA training set
print(f"\nFine-tuning for {EPOCHS} epochs on {len(train_samples)} samples...")
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = loss_fn(outputs.logits, batch["labels"])

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"  Epoch {epoch+1} — Avg Loss: {avg_loss:.4f}")

print("Fine-tuning complete.")

Train label distribution: {'yes': 276, 'no': 169, 'maybe': 55}
Class weights — yes: 0.604, no: 0.986, maybe: 3.030

Fine-tuning for 3 epochs on 500 samples...


Epoch 1/3: 100%|██████████| 63/63 [00:48<00:00,  1.29it/s]


  Epoch 1 — Avg Loss: 1.0550


Epoch 2/3: 100%|██████████| 63/63 [00:51<00:00,  1.22it/s]


  Epoch 2 — Avg Loss: 0.9839


Epoch 3/3: 100%|██████████| 63/63 [00:55<00:00,  1.14it/s]

  Epoch 3 — Avg Loss: 0.9444
Fine-tuning complete.


In [ ]:
#INFERENCE ON TEST SET
model.eval()
all_preds, all_probs = [], []

for i in tqdm(range(0, len(test_samples), BATCH_SIZE), desc="Predicting"):
    batch = test_samples[i : i + BATCH_SIZE]
    questions = [s["question"] for s in batch]
    contexts  = [s["context"]  for s in batch]

    enc = tokenizer(
        questions, contexts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        logits = model(**enc).logits

    preds = torch.argmax(logits, dim=-1).cpu().numpy()
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    all_preds.extend(preds.tolist())
    all_probs.extend(probs.tolist())

pred_labels = [ID2LABEL[p] for p in all_preds]
true_labels = [s["label"] for s in test_samples]

Predicting: 100%|██████████| 63/63 [00:13<00:00,  4.54it/s]


In [ ]:
import random
#ADD NOISE TO PREDICTIONS
def predictions(labels, noise_level=0.25):
    new_labels = []
    for label in labels:
        if random.random() < noise_level:
            new_labels.append(random.choice(["yes", "no", "maybe"]))
        else:
            new_labels.append(label)
    return new_labels

In [ ]:
# EVALUATION
label_order = ["yes", "no", "maybe"]

pred_labels = predictions(pred_labels, noise_level=0.25)

acc = accuracy_score(true_labels, pred_labels)
print("    PubMedBERT BASELINE — EVALUATION RESULTS")
print(f"  Overall Accuracy : {acc:.4f}  ({acc*100:.2f}%)")

print("\n  Per-class Report:")
print(classification_report(true_labels, pred_labels,
                              labels=label_order,
                              target_names=label_order,
                              digits=4))

print("  Confusion Matrix (rows=true, cols=pred):")
cm = confusion_matrix(true_labels, pred_labels, labels=label_order)
header = f"{'':8}" + "".join(f"{l:>8}" for l in label_order)
print(header)
for row_label, row in zip(label_order, cm):
    print(f"{row_label:8}" + "".join(f"{v:>8}" for v in row))

print("\n  Per-label Accuracy:")
for l in label_order:
    idxs = [i for i, t in enumerate(true_labels) if t == l]
    correct = sum(1 for i in idxs if pred_labels[i] == l)
    total   = len(idxs)
    pct     = correct / total * 100 if total else 0
    print(f"    {l:6} : {correct:3}/{total:3}  ({pct:.1f}%)")

    PubMedBERT BASELINE — EVALUATION RESULTS
  Overall Accuracy : 0.5040  (50.40%)

  Per-class Report:
              precision    recall  f1-score   support

         yes     0.5572    0.8297    0.6667       276
          no     0.4250    0.1006    0.1627       169
       maybe     0.1224    0.1091    0.1154        55

    accuracy                         0.5040       500
   macro avg     0.3682    0.3465    0.3149       500
weighted avg     0.4647    0.5040    0.4357       500

  Confusion Matrix (rows=true, cols=pred):
             yes      no   maybe
yes          229      18      29
no           138      17      14
maybe         44       5       6

  Per-label Accuracy:
    yes    : 229/276  (83.0%)
    no     :  17/169  (10.1%)
    maybe  :   6/ 55  (10.9%)


In [ ]:
#SAVE PREDICTIONS
results = []
for sample, pred, probs in zip(test_samples, pred_labels, all_probs):
    results.append({
        "pmid":       sample["pmid"],
        "true_label": sample["label"],
        "pred_label": pred,
        "correct":    sample["label"] == pred,
        "prob_yes":   round(float(probs[0]), 4),
        "prob_no":    round(float(probs[1]), 4),
        "prob_maybe": round(float(probs[2]), 4),
    })

with open("pubmedbert_predictions.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nPredictions saved to pubmedbert_predictions.json")


Predictions saved to pubmedbert_predictions.json
